# Chapter 3: Zones and Graph Queries

This notebook divides the map data into named zones using bounding boxes, assigns each `Intersection` node to a zone and adds `ADJACENT_TO` relationships between neighboring zones. These zones are used later by the simulator and the Streamlit app to dispatch vehicles and report demand.

## 1. Install Dependencies

In [1]:
%pip install neo4j==5.28.1 \
             pyyaml==6.0.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os

from config_validator import load_config, ConfigError, zone_adjacency
from neo4j import GraphDatabase

## 3. Configuration

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

try:
    cfg = load_config("config.yaml")
except (FileNotFoundError, ConfigError) as e:
    raise SystemExit(f"Config error: {e}")

ZONES = [
    {
        "name":    z["name"],
        "lat_min": z["lat_min"],
        "lat_max": z["lat_max"],
        "lon_min": z["lon_min"],
        "lon_max": z["lon_max"],
    }
    for z in cfg["zones"]
]

ADJACENCY = [tuple(pair) for pair in (cfg.get("adjacency") or [])]

print(f"City  : {cfg['city']['name']}")
print(f"Zones : {[z['name'] for z in cfg['zones']]}")
print(f"Pairs : {len(ADJACENCY)} adjacency pairs")
print("Configuration set.")

City  : London Borough of Merton
Zones : ['Wimbledon', 'Raynes Park', 'Colliers Wood', 'Mitcham', 'Morden']
Pairs : 4 adjacency pairs
Configuration set.


## 4. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Neo4j connection verified.")

None
Neo4j connection verified.


## 5. Clear Existing Zone Data

In [5]:
with driver.session() as session:
    session.run("""
        MATCH ()-[r:IN_ZONE]->()
        CALL (r) { DELETE r } IN TRANSACTIONS OF 1000 ROWS
    """)
    session.run("""
        MATCH ()-[r:ADJACENT_TO]->()
        CALL (r) { DELETE r } IN TRANSACTIONS OF 1000 ROWS
    """)
    session.run("""
        MATCH (z:Zone)
        CALL (z) { DETACH DELETE z } IN TRANSACTIONS OF 100 ROWS
    """)

print("Existing zone data cleared.")

Existing zone data cleared.


## 6. Create Zone Nodes

In [6]:
with driver.session() as session:
    session.run("""
        UNWIND $zones AS zone
        MERGE (z:Zone {name: zone.name})
        SET z.lat_min = zone.lat_min,
            z.lat_max = zone.lat_max,
            z.lon_min = zone.lon_min,
            z.lon_max = zone.lon_max
    """, zones=ZONES)

print(f"Created {len(ZONES)} zone nodes.")

Created 5 zone nodes.


## 7. Assign Intersections to Zones

In [7]:
with driver.session() as session:
    # Assign each intersection to the zone whose bounding box contains it
    result = session.run("""
        MATCH (i:Intersection)
        MATCH (z:Zone)
        WHERE i.lat >= z.lat_min AND i.lat < z.lat_max
          AND i.lon >= z.lon_min AND i.lon < z.lon_max
        MERGE (i)-[:IN_ZONE]->(z)
        RETURN count(*) AS assigned
    """)
    assigned = result.single()["assigned"]

    # Assign any remaining intersections to Unknown zone
    session.run("""
        MERGE (z:Zone {name: "Unknown"})
    """)
    result = session.run("""
        MATCH (i:Intersection)
        WHERE NOT (i)-[:IN_ZONE]->()
        MATCH (z:Zone {name: "Unknown"})
        MERGE (i)-[:IN_ZONE]->(z)
        RETURN count(*) AS unassigned
    """)
    unassigned = result.single()["unassigned"]

print(f"Assigned     : {assigned:,} intersections to named zones")
print(f"Unknown zone : {unassigned:,} intersections")

Assigned     : 3,202 intersections to named zones
Unknown zone : 1 intersections


## 8. Add Zone Adjacency Relationships

In [8]:
adjacency_rows = [{"a": a, "b": b} for a, b in ADJACENCY]

with driver.session() as session:
    session.run("""
        UNWIND $pairs AS pair
        MATCH (a:Zone {name: pair.a})
        MATCH (b:Zone {name: pair.b})
        MERGE (a)-[:ADJACENT_TO]->(b)
        MERGE (b)-[:ADJACENT_TO]->(a)
    """, pairs=adjacency_rows)

print(f"Created {len(ADJACENCY) * 2} adjacency relationships ({len(ADJACENCY)} pairs, bidirectional).")

Created 8 adjacency relationships (4 pairs, bidirectional).


## 9. Verify

In [9]:
with driver.session() as session:
    # Intersection count per zone
    result = session.run("""
        MATCH (i:Intersection)-[:IN_ZONE]->(z:Zone)
        RETURN z.name AS zone, count(i) AS intersections
        ORDER BY intersections DESC
    """)
    print("Intersections per zone:")
    total = 0
    for rec in result:
        print(f"  {rec['zone']:<15} {rec['intersections']:,}")
        total += rec['intersections']
    print(f"  {'Total':<15} {total:,}")

    print()

    # Zone adjacency
    result = session.run("""
        MATCH (a:Zone)-[:ADJACENT_TO]->(b:Zone)
        RETURN a.name AS zone, collect(b.name) AS neighbours
        ORDER BY zone
    """)
    print("Zone adjacency:")
    for rec in result:
        print(f"  {rec['zone']:<15} -> {', '.join(sorted(rec['neighbours']))}")

Intersections per zone:
  Wimbledon       839
  Raynes Park     761
  Mitcham         723
  Colliers Wood   544
  Morden          335
  Unknown         1
  Total           3,203

Zone adjacency:
  Colliers Wood   -> Mitcham, Wimbledon
  Mitcham         -> Colliers Wood, Morden
  Morden          -> Mitcham
  Raynes Park     -> Wimbledon
  Wimbledon       -> Colliers Wood, Raynes Park


## 10. Sample Graph Queries

In [10]:
zone_names = [z["name"] for z in cfg["zones"]]
zone_a = zone_names[0]   # first zone
zone_b = zone_names[-1]  # last zone

with driver.session() as session:
    # Which zones can be reached in 1 hop from the first zone?
    result = session.run("""
        MATCH (z:Zone {name: $zone})-[:ADJACENT_TO]->(neighbour:Zone)
        RETURN neighbour.name AS zone
        ORDER BY zone
    """, zone=zone_a)
    neighbours = [rec["zone"] for rec in result]
    print(f"Zones adjacent to {zone_a} : {', '.join(neighbours)}")

    print()

    # Which zones can be reached within 2 hops from the last zone?
    result = session.run("""
        MATCH (z:Zone {name: $zone})-[:ADJACENT_TO*1..2]->(reachable:Zone)
        WHERE reachable.name <> $zone
        RETURN DISTINCT reachable.name AS zone
        ORDER BY zone
    """, zone=zone_b)
    reachable = [rec["zone"] for rec in result]
    print(f"Zones reachable within 2 hops from {zone_b} : {', '.join(reachable)}")

    print()

    # How many intersections in the first zone connect 3 or more roads?
    result = session.run("""
        MATCH (i:Intersection)-[:IN_ZONE]->(z:Zone {name: $zone})
        WHERE i.street_count >= 3
        RETURN count(i) AS complex_intersections
    """, zone=zone_a)
    print(f"Complex intersections in {zone_a} (3+ roads) : {result.single()['complex_intersections']:,}")

Zones adjacent to Wimbledon : Colliers Wood, Raynes Park

Zones reachable within 2 hops from Morden : Colliers Wood, Mitcham

Complex intersections in Wimbledon (3+ roads) : 635


## 11. Teardown

In [11]:
driver.close()
print("Driver closed.")

Driver closed.
